In [1]:
!nvidia-smi

Mon Apr 13 11:18:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%writefile input.txt
Today's weather is sunny and warm. I love sunny days. The sun is shining brightly in the sky 

Writing input.txt


In [3]:
%%writefile wordcount.cu
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>
#include <time.h>
#include <cuda_runtime.h>

#define MAX_WORDS 1000
#define MAX_LEN 20
#define ITERS 1000

// ---------------- GPU KERNEL ----------------
__global__ void countWords(char words[][MAX_LEN], int *counts, int n) {
    int i = threadIdx.x + blockIdx.x * blockDim.x;

    if (i < n) {
        int count = 0;

        for (int j = 0; j < n; j++) {
            int match = 1;

            for (int k = 0; k < MAX_LEN; k++) {
                if (words[i][k] != words[j][k]) {
                    match = 0;
                    break;
                }
                if (words[i][k] == '\0') break;
            }

            if (match) count++;
        }

        counts[i] = count;
    }
}

// ---------------- LOWERCASE FUNCTION ----------------
void toLowerCase(char *word) {
    for (int i = 0; word[i]; i++) {
        word[i] = tolower(word[i]);
    }
}

// ---------------- CPU WORD COUNT ----------------
void countWordsCPU(char words[][MAX_LEN], int *counts, int n) {
    for (int i = 0; i < n; i++) {
        int count = 0;

        for (int j = 0; j < n; j++) {
            int match = 1;

            for (int k = 0; k < MAX_LEN; k++) {
                if (words[i][k] != words[j][k]) {
                    match = 0;
                    break;
                }
                if (words[i][k] == '\0') break;
            }

            if (match) count++;
        }

        counts[i] = count;
    }
}

// ---------------- MAIN ----------------
int main() {
    FILE *fp = fopen("input.txt", "r");
    if (!fp) {
        printf("File not found!\n");
        return 1;
    }

    char words[MAX_WORDS][MAX_LEN];
    int n = 0;

    // -------- READ FILE --------
    while (n < MAX_WORDS && fscanf(fp, "%19s", words[n]) != EOF) {
        toLowerCase(words[n]);
        n++;
    }
    fclose(fp);

    printf("Total words: %d\n", n);

    // -------- GPU MEMORY --------
    char (*d_words)[MAX_LEN];
    int *d_counts;
    int counts[MAX_WORDS];
    int cpu_counts[MAX_WORDS];

    cudaMalloc(&d_words, sizeof(char) * MAX_WORDS * MAX_LEN);
    cudaMalloc(&d_counts, sizeof(int) * MAX_WORDS);

    cudaMemcpy(d_words, words, sizeof(char) * MAX_WORDS * MAX_LEN, cudaMemcpyHostToDevice);

    // -------- LAUNCH KERNEL --------
    int threads = 256;
    int blocks = (n + threads - 1) / threads;

    cudaEvent_t gpuStart, gpuStop;
    cudaEventCreate(&gpuStart);
    cudaEventCreate(&gpuStop);

    cudaEventRecord(gpuStart);
    for (int iter = 0; iter < ITERS; iter++) {
        countWords<<<blocks, threads>>>(d_words, d_counts, n);
    }
    cudaEventRecord(gpuStop);
    cudaEventSynchronize(gpuStop);

    float gpu_time_ms = 0.0f;
    cudaEventElapsedTime(&gpu_time_ms, gpuStart, gpuStop);
    gpu_time_ms /= ITERS;

    cudaMemcpy(counts, d_counts, sizeof(int) * n, cudaMemcpyDeviceToHost);

    // -------- CPU COUNT + TIMING --------
    clock_t cpuStart = clock();
    for (int iter = 0; iter < ITERS; iter++) {
        countWordsCPU(words, cpu_counts, n);
    }
    clock_t cpuStop = clock();

    double cpu_time_ms = (1000.0 * (double)(cpuStop - cpuStart) / CLOCKS_PER_SEC) / ITERS;

    // -------- VALIDATION --------
    int mismatch = 0;
    for (int i = 0; i < n; i++) {
        if (counts[i] != cpu_counts[i]) {
            mismatch = 1;
            break;
        }
    }

    // -------- PRINT DISTINCT WORDS --------
    printf("\nWord Count (GPU):\n");

    for (int i = 0; i < n; i++) {
        int printed = 0;

        for (int j = 0; j < i; j++) {
            if (strcmp(words[i], words[j]) == 0) {
                printed = 1;
                break;
            }
        }

        if (!printed) {
            printf("%s : %d\n", words[i], counts[i]);
        }
    }

    // -------- TIMING COMPARISON --------
    printf("\nAverage GPU time per run: %.6f ms\n", gpu_time_ms);
    printf("Average CPU time per run: %.6f ms\n", cpu_time_ms);
    if (gpu_time_ms > 0.0f) {
        printf("Speedup (CPU/GPU): %.2fx\n", cpu_time_ms / gpu_time_ms);
    }
    printf("Result check: %s\n", mismatch ? "Mismatch" : "Match");

    // -------- CLEANUP --------
    cudaEventDestroy(gpuStart);
    cudaEventDestroy(gpuStop);
    cudaFree(d_words);
    cudaFree(d_counts);

    return 0;
}

Writing wordcount.cu


In [4]:
!nvcc wordcount.cu -o wordcount
!./wordcount

Total words: 18

Word Count (GPU):
today's : 1
weather : 1
is : 2
sunny : 2
and : 1
warm. : 1
i : 1
love : 1
days. : 1
the : 2
sun : 1
shining : 1
brightly : 1
in : 1
sky : 1

Average GPU time per run: 0.128615 ms
Average CPU time per run: 0.002432 ms
Speedup (CPU/GPU): 0.02x
Result check: Match
